# Aristos absolute backtest (ABSOLUTE-BACKTEST-1): Colab fallback

Runs `python -m aristos_council.absolute_backtest run` when the local machine cannot. It is the same program: ten years of
monthly rounds, three forms, holds of 12 and 36 months, no model calls. It spends EODHD units (about 10 per company, one per
bond series), never Anthropic credits.

**Before you start**, copy the folder `data\local\cohorts` from the repo to `MyDrive/aristos/cohorts` (the frozen member lists
are not in GitHub). Results are written to `MyDrive/aristos/backtests_absolute`, one folder per industry, and a finished
industry is never redone, so you can stop the run (or let `BUDGET` stop it) and run the cell again to resume. Set `BUDGET` to
the EODHD units this session may spend; the run stops cleanly between industries when the next one would pass it.

Read `docs/ABSOLUTE_BACKTEST.md` for what the numbers mean. Nothing here is proof: the company list is today's market index
and cannot see companies that failed and vanished.


In [ ]:
#@title Install, key, run (resumes; stops cleanly at the budget)
BRANCH = "main"        #@param {type:"string"}
BUDGET = 30000         #@param {type:"integer"}
import glob, os, subprocess, sys
from getpass import getpass

if not os.path.exists("/content/aristos-council"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/kayvonsalari/aristos-council"], check=True, cwd="/content")
else:
    subprocess.run(["git", "pull"], cwd="/content/aristos-council")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/aristos-council[yfinance]"], check=True)

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/aristos"
OUT = f"{DRIVE}/backtests_absolute"
os.makedirs(OUT, exist_ok=True)
members = glob.glob(f"{DRIVE}/**/chemicals/v*/members.csv", recursive=True)   # find the copied cohorts folder
assert members, "copy data/local/cohorts to MyDrive/aristos/cohorts first"
COHORTS = os.path.dirname(os.path.dirname(os.path.dirname(members[0])))

os.environ["EODHD_API_KEY"] = getpass("EODHD API key: ").strip()      # typed, never saved in the notebook
os.environ["PYTHONUNBUFFERED"] = "1"
os.chdir("/content/aristos-council")

# the cache keeps a same-day re-run free; --budget stops cleanly between industries; a re-run resumes
subprocess.run([sys.executable, "-m", "aristos_council.absolute_backtest", "run", "--budget", str(BUDGET),
                "--out", OUT, "--cohorts-root", COHORTS], check=False)
subprocess.run([sys.executable, "-m", "aristos_council.absolute_backtest", "status", "--out", OUT,
                "--cohorts-root", COHORTS], check=False)
# when every industry is finished: summarise writes SUMMARY.csv and COUNTS.csv next to the industry folders
# !python -m aristos_council.absolute_backtest summarise --out {OUT}
